# 40. The Port Capacity & Expansion Timing Problem
## Tier 5: The Integrated Digital Twin (System-of-Systems Simulation)

### Goal
Transform port capacity planning from static analysis to dynamic system simulation using digital twin technology, enabling real-time optimization and predictive planning through virtual replicas of port infrastructure, operations, and supply chain networks.

### Key assumptions
- Real-time data synchronization between physical and virtual systems
- Multi-system integration across berth operations, yard management, gate processing
- Physics-based simulation of operational dynamics and constraints
- Predictive analytics for demand forecasting and disruption modeling

### Approach (step-by-step)
1. **Digital Twin Architecture**: Create interconnected subsystem models
2. **Real-Time Data Integration**: Ingest sensor data and operational metrics
3. **Physics-Based Simulation**: Model container flows and equipment dynamics
4. **Predictive Analytics**: Forecast demand and identify bottlenecks
5. **What-If Analysis**: Test expansion scenarios in virtual environment
6. **KPI Monitoring**: Track system-wide performance metrics
7. **Scenario Comparison**: Evaluate alternatives with measurable outcomes

### Concrete example (from the source)
Port Meridian digital twin with 847 IoT sensors and 12 operational systems:
- **Current State**: 87.3% berth utilization, 78.4% yard density, 167 trucks/hour gate throughput
- **Scenario Analysis**: Three expansion alternatives tested
- **Results**: Integrated approach achieves 35% higher NPV despite 78% higher investment

### Why this Tier exists vs Tiers 1-4
Previous tiers provide optimization solutions but lack real-time system integration. The digital twin enables dynamic simulation of complex interactions between port subsystems, providing insights that static optimization cannot capture, including emergent behaviors and system-wide impacts of expansion decisions.

### When to use this Tier
- Complex port operations with multiple interconnected systems
- When real-time decision support is required
- For testing expansion scenarios before physical implementation
- When system-wide optimization is more important than individual component optimization

In [ ]:
# Digital Twin Implementation for Port Capacity Planning
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from dataclasses import dataclass
from typing import Dict, List, Tuple
import random
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)
random.seed(42)

plt.style.use('default')
sns.set_palette("husl")

print("Digital Twin system initialized for port capacity planning")

In [ ]:
@dataclass
class DigitalTwinParameters:
    """Parameters for digital twin simulation"""
    current_capacity: float = 3.2  # Million TEU
    current_berths: int = 4
    current_cranes: int = 12
    gate_capacity: int = 200  # Trucks per hour
    yard_capacity: float = 10000  # TEU
    
class PortDigitalTwin:
    """Integrated digital twin for port capacity planning"""
    
    def __init__(self, params: DigitalTwinParameters):
        self.params = params
        self.current_time = datetime(2024, 1, 1)
        
        # Subsystem states
        self.berth_operations = self._initialize_berth_system()
        self.yard_management = self._initialize_yard_system()
        self.gate_processing = self._initialize_gate_system()
        self.intermodal_integration = self._initialize_intermodal_system()
        
        # Performance metrics
        self.kpi_history = []
        
    def _initialize_berth_system(self) -> Dict:
        """Initialize berth operations subsystem"""
        return {
            'berths': self.params.current_berths,
            'cranes': self.params.current_cranes,
            'utilization': 0.873,  # 87.3% average utilization
            'peak_utilization': 0.942,  # 94.2% peak
            'vessel_queue': [],
            'productivity': 25.0  # Moves per hour
        }
    
    def _initialize_yard_system(self) -> Dict:
        """Initialize yard management subsystem"""
        return {
            'capacity': self.params.yard_capacity,
            'utilization': 0.784,  # 78.4% average
            'peak_utilization': 0.917,  # 91.7% peak
            'containers': int(self.params.yard_capacity * 0.784),
            'density': 0.784,
            'equipment': 40,  # Yard cranes
        }
    
    def _initialize_gate_system(self) -> Dict:
        """Initialize gate processing subsystem"""
        return {
            'capacity': self.params.gate_capacity,
            'throughput': 167,  # Trucks per hour average
            'peak_throughput': 234,  # Peak
            'queue_length': 45,  # Average trucks waiting
            'processing_time': 12,  # Minutes per truck
            'efficiency': 0.85,
        }
    
    def _initialize_intermodal_system(self) -> Dict:
        """Initialize intermodal integration subsystem"""
        return {
            'rail_capacity': 500,  # TEU per day
            'truck_capacity': 800,  # TEU per day
            'barge_capacity': 200,  # TEU per day
            'utilization': 0.721,  # 72.1% rail utilization
            'coordination_efficiency': 0.81,
        }
    
    def simulate_expansion_scenario(self, scenario_name: str, 
                                   berth_addition: int = 0,
                                   automation_investment: float = 0,
                                   yard_expansion: float = 0,
                                   duration_days: int = 365) -> Dict:
        """Simulate expansion scenario with digital twin"""
        
        # Store initial state
        initial_state = self._capture_system_state()
        
        # Apply expansion changes
        self._apply_expansion_changes(berth_addition, automation_investment, yard_expansion)
        
        # Run simulation
        simulation_results = []
        
        for day in range(duration_days):
            # Update daily operations
            self._update_daily_operations()
            
            # Capture KPIs
            daily_kpis = self._calculate_kpis()
            daily_kpis['day'] = day
            daily_kpis['scenario'] = scenario_name
            simulation_results.append(daily_kpis)
        
        # Restore initial state
        self._restore_system_state(initial_state)
        
        return {
            'daily_results': simulation_results,
            'summary': self._analyze_simulation_results(simulation_results),
            'scenario_name': scenario_name
        }
    
    def _apply_expansion_changes(self, berth_addition: int, automation_investment: float, yard_expansion: float):
        """Apply expansion changes to digital twin"""
        # Berth expansion
        if berth_addition > 0:
            self.berth_operations['berths'] += berth_addition
            self.berth_operations['cranes'] += berth_addition * 3  # 3 cranes per berth
            # Reduce utilization due to increased capacity
            self.berth_operations['utilization'] *= 0.7
            self.berth_operations['peak_utilization'] *= 0.75
        
        # Automation investment
        if automation_investment > 0:
            automation_level = automation_investment / 1000  # Normalized
            # Improve productivity and efficiency
            self.berth_operations['productivity'] *= (1 + 0.3 * automation_level)
            self.yard_management['equipment'] = int(self.yard_management['equipment'] * (1 + 0.2 * automation_level))
            self.gate_processing['efficiency'] *= (1 + 0.35 * automation_level)
            self.intermodal_integration['coordination_efficiency'] *= (1 + 0.25 * automation_level)
        
        # Yard expansion
        if yard_expansion > 0:
            self.yard_management['capacity'] *= (1 + yard_expansion)
            self.yard_management['utilization'] *= 0.8
            self.yard_management['peak_utilization'] *= 0.85
    
    def _update_daily_operations(self):
        """Update daily operations with realistic variations"""
        # Add daily variations
        daily_variation = np.random.normal(1.0, 0.1)
        
        # Update berth operations
        self.berth_operations['utilization'] = np.clip(
            self.berth_operations['utilization'] * daily_variation, 0.3, 0.98)
        
        # Update yard operations
        self.yard_management['utilization'] = np.clip(
            self.yard_management['utilization'] * daily_variation, 0.3, 0.95)
        
        # Update gate operations
        self.gate_processing['throughput'] = int(
            self.gate_processing['throughput'] * daily_variation)
        
        # Update intermodal
        self.intermodal_integration['utilization'] = np.clip(
            self.intermodal_integration['utilization'] * daily_variation, 0.4, 0.9)
    
    def _calculate_kpis(self) -> Dict:
        """Calculate system-wide KPIs"""
        # System efficiency (weighted average of subsystems)
        berth_efficiency = 1.0 - self.berth_operations['utilization']
        yard_efficiency = 1.0 - self.yard_management['utilization']
        gate_efficiency = self.gate_processing['throughput'] / self.params.gate_capacity
        intermodal_efficiency = self.intermodal_integration['coordination_efficiency']
        
        system_efficiency = (berth_efficiency * 0.3 + yard_efficiency * 0.25 + 
                            gate_efficiency * 0.25 + intermodal_efficiency * 0.2)
        
        # Calculate congestion costs
        congestion_cost = (
            max(0, self.berth_operations['utilization'] - 0.8) * 10000 +
            max(0, self.yard_management['utilization'] - 0.8) * 8000 +
            max(0, self.gate_processing['queue_length'] - 30) * 500
        )
        
        return {
            'system_efficiency': system_efficiency,
            'berth_utilization': self.berth_operations['utilization'],
            'yard_utilization': self.yard_management['utilization'],
            'gate_throughput': self.gate_processing['throughput'],
            'congestion_cost': congestion_cost,
            'total_capacity': self.params.current_capacity * (1 + 0.4 * (self.berth_operations['berths'] - self.params.current_berths)),
        }
    
    def _capture_system_state(self) -> Dict:
        """Capture current system state for restoration"""
        return {
            'berth_operations': self.berth_operations.copy(),
            'yard_management': self.yard_management.copy(),
            'gate_processing': self.gate_processing.copy(),
            'intermodal_integration': self.intermodal_integration.copy(),
        }
    
    def _restore_system_state(self, state: Dict):
        """Restore system state from captured snapshot"""
        self.berth_operations = state['berth_operations'].copy()
        self.yard_management = state['yard_management'].copy()
        self.gate_processing = state['gate_processing'].copy()
        self.intermodal_integration = state['intermodal_integration'].copy()
    
    def _analyze_simulation_results(self, results: List[Dict]) -> Dict:
        """Analyze simulation results and generate summary"""
        df = pd.DataFrame(results)
        
        return {
            'avg_system_efficiency': df['system_efficiency'].mean(),
            'avg_congestion_cost': df['congestion_cost'].mean(),
            'peak_berth_utilization': df['berth_utilization'].max(),
            'avg_gate_throughput': df['gate_throughput'].mean(),
            'total_congestion_cost': df['congestion_cost'].sum(),
            'efficiency_improvement': df['system_efficiency'].mean() - 0.814,  # vs baseline
        }

print("Digital Twin framework defined successfully")

In [ ]:
# Initialize digital twin and run scenario analysis
print("\n" + "="*80)
print("DIGITAL TWIN - PORT CAPACITY EXPANSION SIMULATION")
print("="*80)

# Initialize digital twin
dt_params = DigitalTwinParameters()
digital_twin = PortDigitalTwin(dt_params)

print("\nINITIAL SYSTEM STATE:")
initial_kpis = digital_twin._calculate_kpis()
print(f"System Efficiency: {initial_kpis['system_efficiency']:.1%}")
print(f"Berth Utilization: {initial_kpis['berth_utilization']:.1%}")
print(f"Yard Utilization: {initial_kpis['yard_utilization']:.1%}")
print(f"Gate Throughput: {initial_kpis['gate_throughput']} trucks/hour")
print(f"Daily Congestion Cost: ${initial_kpis['congestion_cost']:,.0f}")

# Define expansion scenarios
scenarios = {
    'Baseline': {
        'berth_addition': 0,
        'automation_investment': 0,
        'yard_expansion': 0,
        'investment_cost': 0
    },
    'Berth Addition': {
        'berth_addition': 2,
        'automation_investment': 0,
        'yard_expansion': 0,
        'investment_cost': 480  # $480M
    },
    'Automation Investment': {
        'berth_addition': 0,
        'automation_investment': 340,
        'yard_expansion': 0,
        'investment_cost': 340  # $340M
    },
    'Integrated Approach': {
        'berth_addition': 2,
        'automation_investment': 340,
        'yard_expansion': 0.2,
        'investment_cost': 720  # $720M
    }
}

print("\nEXPANSION SCENARIOS:")
for name, config in scenarios.items():
    print(f"\n{name}:")
    print(f"  Investment: ${config['investment_cost']}M")
    print(f"  Berth Addition: {config['berth_addition']} berths")
    print(f"  Automation Investment: ${config['automation_investment']}M")
    print(f"  Yard Expansion: {config['yard_expansion']:.0%}")

# Run simulations
print("\n" + "="*60)
print("RUNNING DIGITAL TWIN SIMULATIONS")
print("="*60)

simulation_results = {}
for scenario_name, config in scenarios.items():
    print(f"\nSimulating {scenario_name}...")
    
    result = digital_twin.simulate_expansion_scenario(
        scenario_name=scenario_name,
        berth_addition=config['berth_addition'],
        automation_investment=config['automation_investment'],
        yard_expansion=config['yard_expansion'],
        duration_days=365
    )
    
    simulation_results[scenario_name] = result
    
    summary = result['summary']
    print(f"  Average System Efficiency: {summary['avg_system_efficiency']:.1%}")
    print(f"  Average Daily Congestion Cost: ${summary['avg_congestion_cost']:,.0f}")
    print(f"  Peak Berth Utilization: {summary['peak_berth_utilization']:.1%}")
    print(f"  Efficiency Improvement: {summary['efficiency_improvement']:.1%}")

In [ ]:
# Analyze and compare simulation results
print("\n" + "="*80)
print("DIGITAL TWIN - COMPREHENSIVE RESULTS ANALYSIS")
print("="*80)

# Create comparison table
comparison_data = []

for scenario_name, result in simulation_results.items():
    summary = result['summary']
    config = scenarios[scenario_name]
    
    # Calculate NPV (simplified)
    annual_congestion_savings = (initial_kpis['congestion_cost'] - summary['avg_congestion_cost']) * 365
    npv = annual_congestion_savings * 5 - config['investment_cost']  # 5-year horizon
    roi = (npv / config['investment_cost'] * 100) if config['investment_cost'] > 0 else 0
    
    comparison_data.append({
        'Scenario': scenario_name,
        'Investment ($M)': config['investment_cost'],
        'Avg System Efficiency': f"{summary['avg_system_efficiency']:.1%}",
        'Peak Berth Utilization': f"{summary['peak_berth_utilization']:.1%}",
        'Annual Congestion Savings ($M)': annual_congestion_savings / 1000000,
        '5-Year NPV ($M)': npv / 1000000,
        'ROI (%)': f"{roi:.1f}",
        'Efficiency Improvement': f"{summary['efficiency_improvement']:.1%}"
    })

comparison_df = pd.DataFrame(comparison_data)
display(comparison_df)

# Find best scenario
best_scenario = max(comparison_data, key=lambda x: x['5-Year NPV ($M)'])
print(f"\n🏆 BEST SCENARIO: {best_scenario['Scenario']}")
print(f"   5-Year NPV: ${best_scenario['5-Year NPV ($M)']:.1f}M")
print(f"   ROI: {best_scenario['ROI (%)}%")
print(f"   Efficiency Improvement: {best_scenario['Efficiency Improvement']}")

# Key insights
print("\n" + "="*60)
print("KEY DIGITAL TWIN INSIGHTS:")
print("="*60)

baseline_result = simulation_results['Baseline']['summary']
berth_result = simulation_results['Berth Addition']['summary']
automation_result = simulation_results['Automation Investment']['summary']
integrated_result = simulation_results['Integrated Approach']['summary']

print(f"\n📊 SYSTEM PERFORMANCE COMPARISON:")
print(f"   Baseline Efficiency: {baseline_result['avg_system_efficiency']:.1%}")
print(f"   Berth Addition Efficiency: {berth_result['avg_system_efficiency']:.1%} (+{(berth_result['avg_system_efficiency']/baseline_result['avg_system_efficiency']-1)*100:.1f}%)")
print(f"   Automation Efficiency: {automation_result['avg_system_efficiency']:.1%} (+{(automation_result['avg_system_efficiency']/baseline_result['avg_system_efficiency']-1)*100:.1f}%)")
print(f"   Integrated Efficiency: {integrated_result['avg_system_efficiency']:.1%} (+{(integrated_result['avg_system_efficiency']/baseline_result['avg_system_efficiency']-1)*100:.1f}%)")

print(f"\n💰 CONGESTION COST ANALYSIS:")
print(f"   Baseline Annual Cost: ${baseline_result['total_congestion_cost']/365:,.0f}")
print(f"   Berth Addition Savings: ${(baseline_result['total_congestion_cost']-berth_result['total_congestion_cost'])/365:,.0f}/year")
print(f"   Automation Savings: ${(baseline_result['total_congestion_cost']-automation_result['total_congestion_cost'])/365:,.0f}/year")
print(f"   Integrated Savings: ${(baseline_result['total_congestion_cost']-integrated_result['total_congestion_cost'])/365:,.0f}/year")

print(f"\n⚡ INVESTMENT EFFECTIVENESS:")
berth_roi = (annual_congestion_savings * 5 - 480) / 480 * 100
automation_roi = (annual_congestion_savings * 5 - 340) / 340 * 100
integrated_roi = (annual_congestion_savings * 5 - 720) / 720 * 100

print(f"   Berth Addition ROI: {berth_roi:.1f}%")
print(f"   Automation ROI: {automation_roi:.1f}%")
print(f"   Integrated ROI: {integrated_roi:.1f}%")

if integrated_roi > max(berth_roi, automation_roi):
    print(f"   ✅ Integrated approach delivers highest ROI despite higher investment")
else:
    print(f"   ⚠️  Consider individual investments for better ROI")

In [ ]:
# Create comprehensive visualizations
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Digital Twin Simulation Results - Port Capacity Expansion', fontsize=16, fontweight='bold')

# 1. System Efficiency Comparison
scenarios_list = list(simulation_results.keys())
efficiencies = [result['summary']['avg_system_efficiency'] for result in simulation_results.values()]
colors = ['lightcoral', 'skyblue', 'lightgreen', 'gold']

bars = axes[0, 0].bar(scenarios_list, efficiencies, color=colors)
axes[0, 0].set_title('Average System Efficiency by Scenario')
axes[0, 0].set_ylabel('System Efficiency')
axes[0, 0].tick_params(axis='x', rotation=45)
axes[0, 0].grid(True, alpha=0.3)

# Add efficiency labels
for bar, efficiency in zip(bars, efficiencies):
    axes[0, 0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                    f'{efficiency:.1%}', ha='center', va='bottom')

# 2. Congestion Cost Comparison
congestion_costs = [result['summary']['avg_congestion_cost'] for result in simulation_results.values()]

bars = axes[0, 1].bar(scenarios_list, congestion_costs, color=colors)
axes[0, 1].set_title('Average Daily Congestion Cost')
axes[0, 1].set_ylabel('Cost ($)")
axes[0, 1].tick_params(axis='x', rotation=45)
axes[0, 1].grid(True, alpha=0.3)

# Add cost labels
for bar, cost in zip(bars, congestion_costs):
    axes[0, 1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
                    f'${cost:,.0f}', ha='center', va='bottom')

# 3. Investment vs NPV Scatter Plot
investments = [scenarios[name]['investment_cost'] for name in scenarios_list]
npvs = [data['5-Year NPV ($M)'] for data in comparison_data]

scatter = axes[1, 0].scatter(investments, npvs, s=100, c=range(len(scenarios_list)), 
                          cmap='viridis', alpha=0.7)
axes[1, 0].set_title('Investment vs 5-Year NPV')
axes[1, 0].set_xlabel('Investment ($M)')
axes[1, 0].set_ylabel('5-Year NPV ($M)')
axes[1, 0].grid(True, alpha=0.3)

# Add scenario labels
for i, (inv, npv, name) in enumerate(zip(investments, npvs, scenarios_list)):
    axes[1, 0].annotate(name, (inv, npv), xytext=(5, 5), 
                        textcoords='offset points', fontsize=9)

# 4. Time Series Comparison (System Efficiency over time)
days = range(0, 365, 30)  # Sample every 30 days

for scenario_name, result in simulation_results.items():
    daily_results = result['daily_results']
    efficiency_trend = [daily_results[d]['system_efficiency'] for d in days if d < len(daily_results)]
    axes[1, 1].plot(days[:len(efficiency_trend)], efficiency_trend, 
                    marker='o', label=scenario_name, linewidth=2)

axes[1, 1].set_title('System Efficiency Trend (Sample Days)')
axes[1, 1].set_xlabel('Day of Year')
axes[1, 1].set_ylabel('System Efficiency')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Visualization complete - Digital Twin simulation results displayed")

In [ ]:
# Final recommendations and implementation strategy
print("\n" + "="*80)
print("DIGITAL TWIN - FINAL RECOMMENDATIONS")
print("="*80)

# Extract best scenario details
best_result = simulation_results[best_scenario['Scenario']]
best_config = scenarios[best_scenario['Scenario']]

print("\n🎯 OPTIMAL EXPANSION STRATEGY:")
print(f"   Scenario: {best_scenario['Scenario']}")
print(f"   Total Investment: ${best_config['investment_cost']}M")
print(f"   Expected 5-Year NPV: ${best_scenario['5-Year NPV ($M)']:.1f}M")
print(f"   Return on Investment: {best_scenario['ROI (%)']}%")

print("\n📈 SYSTEM PERFORMANCE IMPROVEMENTS:")
baseline_eff = baseline_result['avg_system_efficiency']
best_eff = best_result['summary']['avg_system_efficiency']
improvement = (best_eff / baseline_eff - 1) * 100

print(f"   System Efficiency: {baseline_eff:.1%} → {best_eff:.1%} (+{improvement:.1f}%)")
print(f"   Peak Berth Utilization: {baseline_result['peak_berth_utilization']:.1%} → {best_result['summary']['peak_berth_utilization']:.1%}")
print(f"   Daily Congestion Cost: ${baseline_result['avg_congestion_cost']:,.0f} → ${best_result['summary']['avg_congestion_cost']:,.0f}")
print(f"   Annual Congestion Savings: ${(baseline_result['avg_congestion_cost']-best_result['summary']['avg_congestion_cost'])*365:,.0f}")

print("\n🔧 IMPLEMENTATION PLAN:")
if best_scenario['Scenario'] == 'Integrated Approach':
    print("   Phase 1 (Months 1-12): Berth construction and foundation work")
    print("   Phase 2 (Months 6-18): Automation system installation and testing")
    print("   Phase 3 (Months 12-24): System integration and optimization")
    print("   Phase 4 (Months 18-36): Performance monitoring and fine-tuning")
elif best_scenario['Scenario'] == 'Berth Addition':
    print("   Phase 1 (Months 1-6): Site preparation and foundation")
    print("   Phase 2 (Months 6-12): Berth construction and crane installation")
    print("   Phase 3 (Months 12-18): Testing and commissioning")
    print("   Phase 4 (Months 18-24): Performance optimization")
elif best_scenario['Scenario'] == 'Automation Investment':
    print("   Phase 1 (Months 1-3): Technology assessment and vendor selection")
    print("   Phase 2 (Months 3-9): Equipment procurement and installation")
    print("   Phase 3 (Months 9-12): System integration and testing")
    print("   Phase 4 (Months 12-24): Performance optimization and training")

print("\n💰 FINANCIAL ANALYSIS:")
payback_period = best_config['investment_cost'] / (best_scenario['Annual Congestion Savings ($M)'])
print(f"   Investment: ${best_config['investment_cost']}M")
print(f"   Annual Savings: ${best_scenario['Annual Congestion Savings ($M)']:.1f}M")
print(f"   Payback Period: {payback_period:.1f} years")
print(f"   5-Year NPV: ${best_scenario['5-Year NPV ($M)']:.1f}M")
print(f"   ROI: {best_scenario['ROI (%)']}%")

print("\n⚠️  RISK MITIGATION STRATEGIES:")
print("   • Implement digital twin monitoring throughout construction")
print("   • Use phased approach to manage financial risk")
print("   • Maintain contingency budget (15% of total investment)")
print("   • Establish performance guarantees with vendors")
print("   • Create training programs for staff on new systems")

print("\n🔄 CONTINUOUS IMPROVEMENT:")
print("   • Maintain digital twin for ongoing optimization")
print("   • Regularly update models with actual performance data")
print("   • Explore additional automation opportunities")
print("   • Monitor competitive landscape and market changes")
print("   • Plan for future capacity expansions based on growth trends")

print("\n📊 DIGITAL TWIN ADVANTAGES:")
print("   ✅ Real-time system simulation and optimization")
print("   ✅ Comprehensive multi-system integration")
print("   ✅ Data-driven decision making with predictive analytics")
print("   ✅ Risk reduction through virtual testing")
print("   ✅ Performance monitoring and continuous improvement")
print("   ✅ Scenario analysis and what-if planning")

print("\n" + "="*80)
print("Digital Twin analysis complete - Optimal strategy identified")
print(f"Recommended: {best_scenario['Scenario']} with ${best_scenario['5-Year NPV ($M)']:.1f}M NPV")
print("="*80)